# 07 · Error Analysis, Explainability & Interactive Demo

**Goals**
1. Load any trained checkpoint
2. Collect and inspect **false positives** and **false negatives** on the test split
3. Grad-CAM visualisations
4. Precision-Recall and ROC curves per model
5. **Interactive widget** - upload your own image and run inference with any saved checkpoint

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from PIL import Image

from src.utils import (
    set_seed, compute_metrics, compute_metrics_per_class,
    plot_confusion_matrix, save_figure, save_results_csv, fig_path, log,
    compute_gradcam,
)
from src.models        import build_model
from src.datasets      import create_dataloaders
from src.augmentations import get_eval_transform

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
REPORTS_DIR = PROJECT_ROOT / 'reports'
CHECKPOINTS = PROJECT_ROOT / 'checkpoints'
NB_NAME     = '07_error_analysis'

CFG = {
    'data_root'   : str(PROJECT_ROOT / 'datasets' / 'raw'),
    'batch_size'  : 32,
    'image_size'  : 224,
    'num_workers' : 2,
    'split_ratios': [0.70, 0.15, 0.15],
    'seed'        : SEED,
}

CLASS_NAMES = ['Human (Real)', 'AI (Fake)']
print('Config ready.')

## 1 - Load checkpoints

Scans `checkpoints/` folder. Each `.pt` file must follow:
`<arch>_<augmentation>_seed<seed>.pt`
(e.g. `resnet18_weak_seed42.pt`).


In [ ]:
# Auto-discover checkpoints from filenames
# Convention: <arch>_<augmentation>_seed<seed>.pt
def _parse_ckpt_name(fname: str):
    '''Infer arch & friendly label from checkpoint filename.'''
    name = fname.replace('.pt', '')
    if name.startswith('vit_b_16'):
        arch = 'vit_b_16'
        rest = name[len('vit_b_16_'):]
    else:
        arch = 'resnet18'
        rest = name[len('resnet18_'):] if name.startswith('resnet18_') else name
    import re
    m = re.search(r'_seed(\d+)$', rest)
    seed = int(m.group(1)) if m else None
    aug = rest[:m.start()] if m else rest
    label = aug.replace('_', ' ').title()
    if seed is not None:
        label += f' (seed{seed})'
    return arch, label


def load_model(ckpt_name: str, model_name: str = 'resnet18') -> torch.nn.Module:
    '''Load a classifier from checkpoints/<ckpt_name>.'''
    path = CHECKPOINTS / ckpt_name
    if not path.exists():
        raise FileNotFoundError(path)
    model = build_model(model_name, num_classes=2, pretrained=False)
    state = torch.load(path, map_location=DEVICE, weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    # Drop unexpected keys (e.g. projection_head from SimCLR)
    model_state = model.state_dict()
    filtered = {k: v for k, v in state.items() if k in model_state}
    if len(filtered) < len(state):
        print(f'  (skipped {len(state) - len(filtered)} unexpected key(s))')
    model.load_state_dict(filtered, strict=False)
    return model.to(DEVICE).eval()


# Build MODELS dict by scanning checkpoints/ folder
MODELS = {}
_errors = []
available = sorted(CHECKPOINTS.glob('*.pt'))
for ckpt_path in available:
    try:
        arch, label = _parse_ckpt_name(ckpt_path.name)
        MODELS[label] = load_model(ckpt_path.name, arch)
        print(f'  {label:35s}  ({ckpt_path.name})')
    except Exception as e:
        _errors.append(f'  {ckpt_path.name}: {e}')
if _errors:
    print(f'\n{len(_errors)} load(s) failed:')
    for err in _errors:
        print(err)
else:
    print(f'\nAll {len(MODELS)} model(s) loaded successfully.')
del _errors, available


## 2 · Build test loader & run inference

In [ ]:
_, _, test_loader = create_dataloaders(CFG)
print(f'Test batches: {len(test_loader)}')

In [ ]:
@torch.no_grad()
def run_inference(model, loader):
    """Return (images_cpu, y_true, y_pred, y_prob) tensors/arrays."""
    all_imgs, all_true, all_pred, all_prob = [], [], [], []
    for imgs, labels in loader:
        logits = model(imgs.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[:, 1]
        preds  = logits.argmax(dim=1)
        all_imgs.append(imgs.cpu())
        all_true.extend(labels.numpy())
        all_pred.extend(preds.cpu().numpy())
        all_prob.extend(probs.cpu().numpy())
    return torch.cat(all_imgs), np.array(all_true), np.array(all_pred), np.array(all_prob)


RESULTS = {}
for label, model in MODELS.items():
    imgs, y_true, y_pred, y_prob = run_inference(model, test_loader)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    RESULTS[label] = dict(imgs=imgs, y_true=y_true, y_pred=y_pred,
                          y_prob=y_prob, metrics=metrics)
    print(f'{label:35s}  Acc={metrics["accuracy"]:.4f}  AUC={metrics["auc"]:.4f}  F1={metrics["f1"]:.4f}')

## 3 · Summary metrics table

In [ ]:
if not RESULTS:
    print('??  RESULTS is empty - no models were loaded.  Skipping summary.')
else:
    rows = []
    for label, res in RESULTS.items():
        m = res['metrics']
        rows.append({
            'Model'         : label,
            'Accuracy'      : f"{m['accuracy']:.4f}",
            'Balanced Acc'  : f"{m['balanced_accuracy']:.4f}",
            'Precision'     : f"{m['precision']:.4f}",
            'Recall'        : f"{m['recall']:.4f}",
            'F1'            : f"{m['f1']:.4f}",
            'Specificity'   : f"{m['specificity']:.4f}",
            'AUC'           : f"{m['auc']:.4f}",
            'Avg Precision' : f"{m['avg_precision']:.4f}",
            'MCC'           : f"{m['mcc']:.4f}",
        })

Loads `07_error_analysis_summary.csv` and `07_predictions.csv`.
Works from saved files — no re-inference needed, even after kernel restart.



In [ ]:
# ── Auto-discover all result CSVs in reports/ ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

csv_patterns = sorted(REPORTS_DIR.glob('*_results.csv')) + \
               sorted(REPORTS_DIR.glob('*_summary.csv')) + \
               sorted(REPORTS_DIR.glob('*_comparison.csv'))

# Also include experiment-specific CSVs (02_vit_results, 03_augmentation_results, etc.)
all_csvs = sorted(REPORTS_DIR.glob('*.csv'))
print(f'? Found {len(all_csvs)} CSV(s) in reports/')
for csv_path in all_csvs:
    df = pd.read_csv(csv_path)
    print(f'\n  ── {csv_path.name} ({len(df)} rows) ──')
    display(df)

# ── Dashboard: grouped bar chart for each CSV with metric columns ──
METRIC_COLS = ['Accuracy', 'Balanced Acc', 'Precision', 'Recall', 'F1',
               'Specificity', 'AUC', 'Avg Precision', 'MCC']

for csv_path in all_csvs:
    df = pd.read_csv(csv_path)
    # Find which metric columns actually exist in this CSV
    avail = [c for c in METRIC_COLS if c in df.columns]
    if len(avail) == 0 or 'Model' not in df.columns:
        continue  # not a model comparison CSV, skip
    if len(df) == 0:
        continue

    model_col = 'Model' if 'Model' in df.columns else df.columns[0]
    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(avail))
    n = len(df)
    w = 0.8 / n
    for i, (_, row) in enumerate(df.iterrows()):
        vals = [float(row[m]) for m in avail]
        offset = (i - (n-1)/2) * w
        bars = ax.bar(x + offset, vals, w, label=row[model_col])
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6)

    ax.set_xticks(x)
    ax.set_xticklabels(avail, rotation=30, ha='right')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.set_title(f'Model comparison — {csv_path.name}', fontsize=13)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

if not all_csvs:
    print('No CSVs found in reports/. Run experiments first.')


## 4 · Confusion matrices

In [ ]:
if not RESULTS:
    print('??  No results - skipping.')
else:
    for label, res in RESULTS.items():
        plot_confusion_matrix(
            res['y_true'], res['y_pred'],
            class_names=CLASS_NAMES,
            title=f'Confusion Matrix - {label}',
            path=fig_path(str(REPORTS_DIR), NB_NAME,
                          f'cm_{label.replace(" ","_")}.png'),
            show=True,
        )

## 5 · ROC & Precision-Recall curves (all models)

In [ ]:
from sklearn.metrics import (roc_curve, precision_recall_curve,
                              roc_auc_score, average_precision_score)

if not RESULTS:
    print('??  No results - skipping.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for label, res in RESULTS.items():
        fpr, tpr, _ = roc_curve(res['y_true'], res['y_prob'])
        auc = roc_auc_score(res['y_true'], res['y_prob'])
        axes[0].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})')

        prec, rec, _ = precision_recall_curve(res['y_true'], res['y_prob'])
        ap = average_precision_score(res['y_true'], res['y_prob'])
        axes[1].plot(rec, prec, label=f'{label} (AP={ap:.3f})')

    axes[0].plot([0,1],[0,1],'k--',linewidth=0.8)
    axes[0].set(title='ROC Curves', xlabel='FPR', ylabel='TPR')
    axes[0].legend(fontsize=8)
    axes[1].set(title='Precision-Recall Curves', xlabel='Recall', ylabel='Precision')
    axes[1].legend(fontsize=8)
    fig.suptitle('All Models - ROC & PR', fontsize=13)
    fig.tight_layout()
    save_figure(fig, fig_path(str(REPORTS_DIR), NB_NAME, 'roc_pr_all.png'))

## 6 · False Positives / False Negatives gallery

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denorm(t: torch.Tensor) -> np.ndarray:
    """CHW normalised tensor - HWC uint8 numpy array."""
    t = t.cpu() * STD + MEAN
    return (t.permute(1,2,0).clamp(0,1).numpy() * 255).astype(np.uint8)


def show_error_gallery(imgs, y_true, y_pred, y_prob,
                        error_type='FP', n_show=8,
                        title_prefix='', save_path=None):
    """Plot a grid of misclassified images (FP or FN)."""
    if error_type == 'FP':
        mask   = (y_true == 0) & (y_pred == 1)
        ylabel = 'True: Real  |  Pred: AI'
    else:
        mask   = (y_true == 1) & (y_pred == 0)
        ylabel = 'True: AI  |  Pred: Real'
    idxs = np.where(mask)[0]
    n    = min(n_show, len(idxs))
    if n == 0:
        print(f'No {error_type} errors found for {title_prefix}.')
        return
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, idxs[:n]):
        ax.imshow(denorm(imgs[idx]))
        ax.set_title(f'P(fake)={y_prob[idx]:.2f}', fontsize=7)
        ax.axis('off')
    fig.suptitle(f'{title_prefix} - {error_type}: {ylabel}  ({len(idxs)} total)',
                 fontsize=9)
    fig.tight_layout()
    if save_path:
        save_figure(fig, save_path, show=True)
    else:
        plt.show()


if not RESULTS:
    print('??  No results - skipping.')
else:
    for label, res in RESULTS.items():
        for etype in ('FP', 'FN'):
            show_error_gallery(
                res['imgs'], res['y_true'], res['y_pred'], res['y_prob'],
                error_type=etype, n_show=8, title_prefix=label,
                save_path=fig_path(str(REPORTS_DIR), NB_NAME,
                                   f'{etype}_{label.replace(" ","_")}.png'),
            )

## 7 · Grad-CAM explainability

For **ResNet18** the target layer is `model.layer4[-1]`.  
For ViT or other architectures without `layer4` the panel is skipped automatically.

In [ ]:
import torch.nn as nn
from scipy.ndimage import zoom
from matplotlib.colors import Normalize

def get_gradcam_layer(model: nn.Module):
    """Return the last conv block suitable for Grad-CAM, or None."""
    if hasattr(model, 'layer4'):
        return model.layer4[-1]
    return None


def gradcam_panel(model, imgs, y_true, y_pred, y_prob,
                  n_tp=4, n_fp=4, n_fn=4, label='', save_path=None):
    """Grid of Grad-CAM overlays: rows = [TP, FP, FN]."""
    target_layer = get_gradcam_layer(model)
    if target_layer is None:
        print(f'[{label}] Grad-CAM not supported - skipping.')
        return

    groups = [
        ('TP (AI correct)',  (y_true==1)&(y_pred==1), 1, n_tp),
        ('FP (Real - AI)',   (y_true==0)&(y_pred==1), 1, n_fp),
        ('FN (AI - Real)',   (y_true==1)&(y_pred==0), 0, n_fn),
    ]
    total_cols = max(n_tp, n_fp, n_fn)
    fig, axes  = plt.subplots(len(groups)*2, total_cols,
                               figsize=(total_cols*2, len(groups)*4))
    axes = np.atleast_2d(axes)

    for rp, (row_label, mask, tcls, n_show) in enumerate(groups):
        idxs    = np.where(mask)[0][:n_show]
        row_img = rp * 2
        row_cam = rp * 2 + 1
        for col, idx in enumerate(idxs):
            axes[row_img, col].imshow(denorm(imgs[idx]))
            axes[row_img, col].set_title(f'P={y_prob[idx]:.2f}', fontsize=7)
            axes[row_img, col].axis('off')
            try:
                hm = compute_gradcam(model,
                                     imgs[idx].unsqueeze(0).to(DEVICE),
                                     tcls, target_layer)
                h, w = imgs[idx].shape[1], imgs[idx].shape[2]
                hm   = zoom(hm, (h/hm.shape[0], w/hm.shape[1]))
                axes[row_cam, col].imshow(denorm(imgs[idx]))
                axes[row_cam, col].imshow(hm, alpha=0.5, cmap='jet',
                                          norm=Normalize(0,1))
            except Exception as e:
                axes[row_cam, col].text(0.5, 0.5, str(e),
                                        ha='center', va='center', fontsize=6)
            axes[row_cam, col].axis('off')
        for col in range(len(idxs), total_cols):
            axes[row_img, col].axis('off')
            axes[row_cam, col].axis('off')
        axes[row_img, 0].set_ylabel(row_label, fontsize=8, rotation=0,
                                    labelpad=80, va='center')
    fig.suptitle(f'Grad-CAM - {label}', fontsize=12)
    fig.tight_layout()
    if save_path:
        save_figure(fig, save_path, show=True)
    else:
        plt.show()


if not RESULTS:
    print('??  No results - skipping.')
else:
    for label, res in RESULTS.items():
        gradcam_panel(
            MODELS[label],
            res['imgs'], res['y_true'], res['y_pred'], res['y_prob'],
            n_tp=4, n_fp=4, n_fn=4, label=label,
            save_path=fig_path(str(REPORTS_DIR), NB_NAME,
                               f'gradcam_{label.replace(" ","_")}.png'),
        )

## 8 · Per-class report

In [ ]:
if not RESULTS:
    print('??  No results - skipping.')
else:
    for label, res in RESULTS.items():
        print(f'\n??? {label} ???')
        display(compute_metrics_per_class(res['y_true'], res['y_pred']))

---
## 9 · - Interactive Demo - upload your own image

Run the cell below, **upload an image** (PNG/JPG/WebP), choose a model, then press **? Run**.

The widget shows:
- the uploaded image at 224·224;
- a horizontal confidence bar (`Human` vs `AI`);
- the Grad-CAM overlay *(ResNet-based models only)*.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display as ipy_display, clear_output
from scipy.ndimage import zoom as ndimage_zoom
from matplotlib.colors import Normalize

EVAL_TF = get_eval_transform(image_size=224)

uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp', multiple=False,
    description='? Upload',
    layout=widgets.Layout(width='160px'),
)
model_dd = widgets.Dropdown(
    options=list(MODELS.keys()) if MODELS else ['(no checkpoints loaded)'],
    description='Model:',
    layout=widgets.Layout(width='340px'),
)
run_btn  = widgets.Button(
    description='? Run', button_style='primary',
    layout=widgets.Layout(width='120px'),
)
out = widgets.Output()

def _on_run(_):
    with out:
        clear_output(wait=True)
        if not uploader.value:
            print('??  Upload an image first.'); return
        if not MODELS:
            print('??  No models loaded - run section 1 first.'); return
        chosen = model_dd.value
        if chosen not in MODELS:
            print(f'??  {chosen!r} not in MODELS.'); return

        # decode upload
        # Handle both ipywidgets 7.x (tuple) and 8.x (dict) upload format
        val = uploader.value
        if isinstance(val, dict):
            raw = next(iter(val.values()))['content']
        else:
            raw = val[0]['content']
        pil    = Image.open(io.BytesIO(raw)).convert('RGB')
        tensor = EVAL_TF(pil).unsqueeze(0).to(DEVICE)

        # forward pass
        mdl = MODELS[chosen]
        with torch.no_gradi():
            probs = torch.softmax(mdl(tensor), dim=1)[0].cpu().numpy()
        pred_idx   = int(probs.argmax())
        pred_label = CLASS_NAMES[pred_idx]
        conf       = probs[pred_idx] * 100

        # layout
        cam_layer = get_gradcam_layer(mdl)
        ncols     = 3 if cam_layer else 2
        fig, axes = plt.subplots(1, ncols, figsize=(ncols * 4, 4))
        axes      = np.atleast_1d(axes)
        img_np    = np.array(pil.resize((224, 224)))

        # panel 1 - uploaded image
        axes[0].imshow(img_np)
        axes[0].set_title('Uploaded image', fontsize=10); axes[0].axis('off')

        # panel 2 - confidence bars
        bars = axes[1].barh(CLASS_NAMES, probs*100, color=['steelblue','tomato'])
        axes[1].set_xlim(0, 100)
        axes[1].set_xlabel('Confidence (%)')
        axes[1].set_title(f'{pred_label}  ({conf:.1f}%)', fontsize=10)
        for bar, p in zip(bars, probs):
            axes[1].text(bar.get_width()+1,
                         bar.get_y()+bar.get_height()/2,
                         f'{p*100:.1f}%', va='center', fontsize=9)

        # panel 3 - Grad-CAM (ResNet only)
        if cam_layer:
            try:
                cam_input = tensor.clone().requires_grad_(True)
                hm = compute_gradcam(mdl, cam_input, pred_idx, cam_layer)
                hm = ndimage_zoom(hm, (224/hm.shape[0], 224/hm.shape[1]))
                axes[2].imshow(img_np)
                axes[2].imshow(hm, alpha=0.5, cmap='jet', norm=Normalize(0,1))
                axes[2].set_title('Grad-CAM', fontsize=10)
            except Exception as e:
                axes[2].text(0.5, 0.5, f'CAM error:\n{e}',
                             ha='center', va='center', fontsize=8)
            axes[2].axis('off')

        fig.suptitle(f'Model: {chosen}', fontsize=12)
        fig.tight_layout(); plt.show()

run_btn.on_click(_on_run)

header   = widgets.HTML(
    '<h3 style="margin:0 0 4px">?? Interactive Inference Demo</h3>'
    '<p style="color:#666;font-size:.9em;margin:0 0 10px">'
    'Upload any artwork - the model will classify it as '
    '<b>Human (Real)</b> or <b>AI (Fake)</b> and show where it looked.</p>'
)
controls = widgets.HBox([uploader, model_dd, run_btn],
                         layout=widgets.Layout(gap='10px', align_items='center'))
ipy_display(widgets.VBox(
    [header, controls, out],
    layout=widgets.Layout(padding='14px', border='1px solid #ccc',
                          border_radius='8px', width='max-content'),
))

---
*End of notebook 07 - Error Analysis, Explainability & Interactive Demo*